# Gaussian simulation

This document presents a well-calibrated simulation study for testing the sBayes clustering algorithm. We simulate parameters by drawing samples from the prior distribution, generate synthetic data from these, and pass the data to the sBayes algorithm to infer the simulated parameters. We then evaluate the calibration of the inference procedure by comparing the inferred posterior distributions to the true parameter values.

We use the Gemini LLM to create an empty structure of the synthetic data using the following prompt:

Create a CSV with 20 rows and the following columns:

    name: any first names you can think of
    id: abbreviate the first names to a unique id with three upper case letters
    x: a random longitude
    y: a random latitude
    confounder_1: assign each row randomly to A or B
    f1: keep empty
    f2: keep empty
    ...
    f30: keep empty

In [1]:
from sbayes.experiment_setup import Experiment
from sbayes.load_data import Data as Structure, Data
from sbayes.mcmc_setup import MCMCSetup
from sbayes.sampling.loggers import write_samples
from sbayes.tools.simulation import prepare_folder, write_data, read_parameters, find_title, plot_simulated_against_inferred

from numpyro.infer import Predictive
import jax.random as random
import numpy as np
import pandas as pd
import shutil
import matplotlib.pyplot as plt

We set up the model using the ``config.yaml`` file. This file specifies the number of simulated clusters and confounders, and defines the data type for each feature. In this experiment, all features are discrete count data following a Poisson distribution.

In [2]:
# Initialize the experiment
experiment = Experiment(
    config_file="config.yaml",
    experiment_name="poisson",
)

# Enabling sampling from the prior
experiment.config.model.sample_from_prior = True

# Load the model structure (number of observations, variables, confounders, clusters)
structure = Structure.from_experiment(experiment)

# Set up Model
setup = MCMCSetup(structure, experiment)
model = setup.model.get_model

# We don't need the usual subfolders for this simulation
shutil.rmtree(experiment.path_results)

# NA values?

Experiment: poisson
File location for results: /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/poisson
Start time and date: 22:12:28 26.08.2025


DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/template_data/features.csv.
Gaussian: 40 feature(s) with 8000 NA value(s).


We draw 100 independent sets of parameters from the prior distribution. For each set, we generate a corresponding synthetic dataset.

In [3]:
rng_key = random.PRNGKey(0)
num_samples = 100

# Set up Predictive to draw from prior
predictive = Predictive(model, num_samples=num_samples)

# Sample parameters and synthetic data from prior
prior_samples = predictive(rng_key)

We write the sampled parameters and corresponding synthetic data to file.


In [4]:
empty_features_csv = pd.read_csv(experiment.config.data.features)
results_folder = experiment.config.results.path

# Write samples and data to file
for s in range(num_samples):

    params_folder, data_folder = prepare_folder(results_folder, s)

    i_sample = {k: v[s:s+1] for k, v in prior_samples.items()}

    write_samples(run=0, base_path=params_folder,
                  samples=i_sample,
                  data=structure, model=setup.model)

    write_data(partitions=structure.features.partitions,
               sample=i_sample,features_csv=empty_features_csv.copy(deep=True),
               base_path=data_folder)

Next, for each of the 100 synthetic datasets, we perform inference to recover the corresponding set of sampled parameters.

In [5]:
# Run inference
for s in range(num_samples):

    experiment.config.model.sample_from_prior = False
    experiment.config.data.features = results_folder / f"sim_{s}/sim_data/features.csv"
    experiment.path_results = results_folder / f"sim_{s}/results"
    experiment.path_results.mkdir(parents=False, exist_ok=True)

    # Load the data
    data = Data.from_experiment(experiment)
    # Set up Model
    mcmc = MCMCSetup(data, experiment)
    mcmc.sample(resume=False)




DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/sim_0/sim_data/features.csv.
Gaussian: 40 feature(s) with 0 NA value(s).
100%|██████████| 3/3 [01:33<00:00, 31.16s/it]
Writing samples to disk


Runtime sample_nuts: 197.07s


Runtime: 197.91 seconds


DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/sim_1/sim_data/features.csv.
Gaussian: 40 feature(s) with 0 NA value(s).
100%|██████████| 3/3 [01:29<00:00, 29.93s/it]
Writing samples to disk


Runtime sample_nuts: 178.65s


Runtime: 179.64 seconds


DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/sim_2/sim_data/features.csv.
Gaussian: 40 feature(s) with 0 NA value(s).
100%|██████████| 3/3 [01:27<00:00, 29.12s/it]
Writing samples to disk


...

Runtime: 397.05 seconds


DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/sim_98/sim_data/features.csv.
Gaussian: 40 feature(s) with 0 NA value(s).
100%|██████████| 3/3 [05:41<00:00, 113.74s/it]
Writing samples to disk


Runtime sample_nuts: 555.68s


Runtime: 556.46 seconds


DATA IMPORT
##########################################
200 objects with 40 features read from /home/peter/Desktop/sBayes/sBayes/experiments/simulation/gaussian/sims/sim_99/sim_data/features.csv.
Gaussian: 40 feature(s) with 0 NA value(s).
100%|██████████| 3/3 [01:31<00:00, 30.42s/it]
Writing samples to disk


Runtime sample_nuts: 185.75s


Runtime: 186.52 seconds


For each of the 100 inference runs, we read in the posterior distribution over the parameters.


In [6]:
results_folder = experiment.config.results.path

parameters = read_parameters(
    results_folder, k=2,
    feature_names=structure.features.names,
    confounder_names={k: v.group_names for k, v in structure.confounders.items()}
)

We plot the simulated (true) parameters against the inferred posteriors. We expect that, on average, the true parameter values fall within the 95% credible intervals of the posterior distributions approximately 95% of the time.


In [7]:
column_names_sim = next(iter(parameters.values()))['simulated'].columns.tolist()

for n in column_names_sim:

    if n in ['Sample']:
        pass
    else:
        p_sim = np.array([v['simulated'][n][0] for v in parameters.values()])
        p_inf = np.array([v['inferred'][n] for v in parameters.values()])
        title_plot = find_title(n, structure.confounders, structure.features.names)
        plot_simulated_against_inferred(simulated=p_sim, inferred=p_inf,
                                        title=title_plot)
        plot_folder = results_folder.parent / "plots"
        plot_folder.mkdir(parents=False, exist_ok=True)
        plt.savefig(plot_folder / f"{n}.png")
        plt.close()
